# 🕵️ Hallucination Detective

**A RAG-based pipeline that catches LLM hallucinations in real time.**

Stack: `Claude API` · `ChromaDB` · `sentence-transformers` · `FastAPI` · `Streamlit`

### How it works
1. You give it a **knowledge base** (source documents) — this gets chunked and embedded into ChromaDB.
2. You give it a **claim** (e.g. an LLM's output, or any statement you want fact-checked).
3. The pipeline **retrieves** the most relevant chunks from the knowledge base.
4. Claude acts as a **judge**, comparing the claim against the retrieved evidence and returning a verdict: `GROUNDED`, `HALLUCINATED`, or `UNCERTAIN` — with reasoning and cited evidence.

### Notebook structure
- **Part 1** — Setup & installs
- **Part 2** — RAG pipeline (chunking, embedding, ChromaDB retrieval)
- **Part 3** — Hallucination judge (Claude API two-stage verdict)
- **Part 4** — Quick inline test (no servers needed)
- **Part 5** — FastAPI backend (runs in background inside Colab)
- **Part 6** — Streamlit frontend (tunneled out of Colab so you get a live demo link)

> 💡 You can stop after Part 4 if you just want the core pipeline working. Parts 5–6 are for the full demo-able app.

## Part 1 — Setup

In [ ]:
!pip install -q anthropic chromadb sentence-transformers fastapi uvicorn pyngrok nest-asyncio python-multipart streamlit

In [ ]:
import os
from getpass import getpass

# Paste your Anthropic API key here, or leave blank to be prompted securely.
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

print("API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Part 2 — RAG Pipeline

Chunking + embedding + ChromaDB retrieval. This is the retrieval layer that grounds the judge in real evidence.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
import uuid
import re


def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """Split text into overlapping word-based chunks."""
    words = text.split()
    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
        if i + chunk_size >= len(words):
            break
    return chunks


class KnowledgeBase:
    """Wraps ChromaDB + sentence-transformers for semantic retrieval."""

    def __init__(self, collection_name: str = "knowledge_base", reset: bool = True):
        self.client = chromadb.Client()  # in-memory for Colab; swap for PersistentClient for disk persistence
        self.embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        if reset:
            try:
                self.client.delete_collection(collection_name)
            except Exception:
                pass
        self.collection = self.client.get_or_create_collection(
            name=collection_name, embedding_function=self.embed_fn
        )

    def add_document(self, text: str, source: str = "unknown", chunk_size: int = 500, overlap: int = 50):
        chunks = chunk_text(text, chunk_size=chunk_size, overlap=overlap)
        ids = [str(uuid.uuid4()) for _ in chunks]
        metadatas = [{"source": source, "chunk_index": i} for i in range(len(chunks))]
        if chunks:
            self.collection.add(documents=chunks, ids=ids, metadatas=metadatas)
        return len(chunks)

    def retrieve(self, query: str, top_k: int = 4) -> list[dict]:
        results = self.collection.query(query_texts=[query], n_results=top_k)
        evidence = []
        docs = results.get("documents", [[]])[0]
        metas = results.get("metadatas", [[]])[0]
        dists = results.get("distances", [[]])[0]
        for doc, meta, dist in zip(docs, metas, dists):
            evidence.append({
                "text": doc,
                "source": meta.get("source", "unknown"),
                "chunk_index": meta.get("chunk_index", -1),
                "relevance_score": round(1 - dist, 4),  # cosine distance -> similarity
            })
        return evidence

    def count(self) -> int:
        return self.collection.count()


print("RAG pipeline classes defined.")

## Part 3 — Hallucination Judge

Two-stage design:
1. **Retrieve** evidence for the claim from the knowledge base.
2. **Judge** — Claude compares the claim against retrieved evidence only (not its own parametric knowledge) and returns a structured verdict.

The judge is instructed to say `UNCERTAIN` when evidence is insufficient, rather than guessing — that's the difference between a real fact-checker and a model that just sounds confident either way.

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

JUDGE_SYSTEM_PROMPT = """You are a strict fact-checking judge. You will be given a CLAIM and a set of EVIDENCE chunks retrieved from a knowledge base.

Your job: decide whether the CLAIM is supported by the EVIDENCE.

Rules:
- Base your verdict ONLY on the provided evidence. Ignore anything you personally know that isn't in the evidence.
- If the evidence clearly supports the claim, verdict is GROUNDED.
- If the evidence clearly contradicts the claim, verdict is HALLUCINATED.
- If the evidence is missing, unrelated, or insufficient to decide either way, verdict is UNCERTAIN. Do not guess.
- Quote the specific evidence (by chunk source) that drove your decision.

Respond ONLY with valid JSON, no markdown fences, no preamble, matching exactly this schema:
{
  "verdict": "GROUNDED" | "HALLUCINATED" | "UNCERTAIN",
  "confidence": <float 0-1>,
  "reasoning": "<2-3 sentence explanation>",
  "supporting_evidence": ["<short quote or paraphrase from evidence>", ...]
}"""


def judge_claim(claim: str, evidence: list[dict], model: str = "claude-sonnet-4-6") -> dict:
    if not evidence:
        return {
            "verdict": "UNCERTAIN",
            "confidence": 0.0,
            "reasoning": "No evidence was retrieved from the knowledge base for this claim.",
            "supporting_evidence": [],
        }

    evidence_block = "\n\n".join(
        f"[Evidence {i+1} | source: {e['source']} | relevance: {e['relevance_score']}]\n{e['text']}"
        for i, e in enumerate(evidence)
    )

    user_prompt = f"CLAIM:\n{claim}\n\nEVIDENCE:\n{evidence_block}"

    response = client.messages.create(
        model=model,
        max_tokens=600,
        system=JUDGE_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(json)?|```$", "", raw, flags=re.MULTILINE).strip()

    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        result = {
            "verdict": "UNCERTAIN",
            "confidence": 0.0,
            "reasoning": f"Judge returned unparseable output: {raw[:200]}",
            "supporting_evidence": [],
        }

    result["evidence_used"] = evidence
    return result


def check_claim(kb: "KnowledgeBase", claim: str, top_k: int = 4, model: str = "claude-sonnet-4-6") -> dict:
    """End-to-end: retrieve + judge."""
    evidence = kb.retrieve(claim, top_k=top_k)
    return judge_claim(claim, evidence, model=model)


print("Judge defined.")

## Part 4 — Quick Inline Test

Build a tiny knowledge base and run a few claims through it, no servers required. Good for sanity-checking the pipeline before wiring up FastAPI/Streamlit.

In [ ]:
kb = KnowledgeBase(reset=True)

sample_doc = """
The Eiffel Tower was completed in 1889 as the entrance arch for the 1889 World's Fair in Paris.
It was designed by the engineer Gustave Eiffel's company. The tower stands 330 meters tall,
making it the tallest structure in Paris. It was the tallest man-made structure in the world
until the Chrysler Building in New York was completed in 1930. The tower is made of wrought iron
and weighs approximately 10,100 tons. It receives about 7 million visitors per year, making it
one of the most visited paid monuments in the world. The tower was originally intended to be
a temporary structure and was almost demolished in 1909, but was kept because it proved valuable
for communication purposes, particularly radio transmission.
"""

n_chunks = kb.add_document(sample_doc, source="eiffel_tower_facts.txt")
print(f"Indexed {n_chunks} chunks. Collection size: {kb.count()}")

In [ ]:
test_claims = [
    "The Eiffel Tower was completed in 1889.",                      # GROUNDED
    "The Eiffel Tower is 500 meters tall.",                          # HALLUCINATED (actual: 330m)
    "The Eiffel Tower was designed by Gustave Eiffel's company.",    # GROUNDED
    "The Eiffel Tower has a rotating restaurant at the top.",        # UNCERTAIN/HALLUCINATED (not in evidence)
]

for claim in test_claims:
    result = check_claim(kb, claim)
    print(f"CLAIM: {claim}")
    print(f"  → {result['verdict']}  (confidence: {result['confidence']})")
    print(f"  reasoning: {result['reasoning']}")
    print()

## Part 5 — FastAPI Backend

Writes `app.py` to disk and runs it in a background thread inside Colab so it's reachable on `localhost:8000`. The Streamlit frontend in Part 6 calls this API.

In [ ]:
%%writefile app.py
import os
import re
import json
import uuid

import chromadb
from chromadb.utils import embedding_functions
import anthropic
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="Hallucination Detective API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

client = anthropic.Anthropic()
chroma_client = chromadb.Client()
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = chroma_client.get_or_create_collection(name="knowledge_base", embedding_function=embed_fn)

JUDGE_SYSTEM_PROMPT = """You are a strict fact-checking judge. You will be given a CLAIM and a set of EVIDENCE chunks retrieved from a knowledge base.

Your job: decide whether the CLAIM is supported by the EVIDENCE.

Rules:
- Base your verdict ONLY on the provided evidence. Ignore anything you personally know that isn't in the evidence.
- If the evidence clearly supports the claim, verdict is GROUNDED.
- If the evidence clearly contradicts the claim, verdict is HALLUCINATED.
- If the evidence is missing, unrelated, or insufficient to decide either way, verdict is UNCERTAIN. Do not guess.
- Quote the specific evidence (by chunk source) that drove your decision.

Respond ONLY with valid JSON, no markdown fences, no preamble, matching exactly this schema:
{
  "verdict": "GROUNDED" | "HALLUCINATED" | "UNCERTAIN",
  "confidence": <float 0-1>,
  "reasoning": "<2-3 sentence explanation>",
  "supporting_evidence": ["<short quote or paraphrase from evidence>", ...]
}"""


def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50):
    words = text.split()
    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
        if i + chunk_size >= len(words):
            break
    return chunks


class IngestRequest(BaseModel):
    text: str
    source: str = "uploaded_doc"


class CheckRequest(BaseModel):
    claim: str
    top_k: int = 4


@app.get("/health")
def health():
    return {"status": "ok", "documents_indexed": collection.count()}


@app.post("/ingest")
def ingest(req: IngestRequest):
    chunks = chunk_text(req.text)
    if not chunks:
        return {"chunks_added": 0, "total_documents": collection.count()}
    ids = [str(uuid.uuid4()) for _ in chunks]
    metadatas = [{"source": req.source, "chunk_index": i} for i in range(len(chunks))]
    collection.add(documents=chunks, ids=ids, metadatas=metadatas)
    return {"chunks_added": len(chunks), "total_documents": collection.count()}


@app.post("/check")
def check(req: CheckRequest):
    results = collection.query(query_texts=[req.claim], n_results=req.top_k)
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    dists = results.get("distances", [[]])[0]

    evidence = [
        {
            "text": doc,
            "source": meta.get("source", "unknown"),
            "relevance_score": round(1 - dist, 4),
        }
        for doc, meta, dist in zip(docs, metas, dists)
    ]

    if not evidence:
        return {
            "verdict": "UNCERTAIN",
            "confidence": 0.0,
            "reasoning": "No evidence was retrieved from the knowledge base for this claim.",
            "supporting_evidence": [],
            "evidence_used": [],
        }

    evidence_block = "\n\n".join(
        f"[Evidence {i+1} | source: {e['source']} | relevance: {e['relevance_score']}]\n{e['text']}"
        for i, e in enumerate(evidence)
    )
    user_prompt = f"CLAIM:\n{req.claim}\n\nEVIDENCE:\n{evidence_block}"

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=600,
        system=JUDGE_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(json)?|```$", "", raw, flags=re.MULTILINE).strip()

    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        result = {
            "verdict": "UNCERTAIN",
            "confidence": 0.0,
            "reasoning": f"Judge returned unparseable output: {raw[:200]}",
            "supporting_evidence": [],
        }

    result["evidence_used"] = evidence
    return result

In [ ]:
import nest_asyncio
import uvicorn
import threading
import time

nest_asyncio.apply()

def run_api():
    uvicorn.run("app:app", host="0.0.0.0", port=8000, log_level="warning")

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
time.sleep(3)
print("FastAPI running on http://localhost:8000")

In [ ]:
# Sanity check the API is alive
import requests
print(requests.get("http://localhost:8000/health").json())

## Part 6 — Streamlit Frontend

Writes `streamlit_app.py`, launches it, and tunnels it out of Colab via `localtunnel` (no auth token needed, unlike ngrok). You'll get a public URL to click.

In [ ]:
%%writefile streamlit_app.py
import streamlit as st
import requests

API_URL = "http://localhost:8000"

st.set_page_config(page_title="Hallucination Detective", page_icon="🕵️", layout="centered")

st.title("🕵️ Hallucination Detective")
st.caption("RAG-based pipeline that catches LLM hallucinations in real time.")

with st.expander("📚 Step 1 — Add source documents to the knowledge base", expanded=True):
    doc_source = st.text_input("Source name", value="my_document.txt")
    doc_text = st.text_area("Paste the ground-truth text here", height=200,
                             placeholder="Paste the reference document, article, or facts you want to fact-check claims against...")
    if st.button("Ingest document"):
        if doc_text.strip():
            resp = requests.post(f"{API_URL}/ingest", json={"text": doc_text, "source": doc_source})
            data = resp.json()
            st.success(f"Added {data['chunks_added']} chunks. Total indexed: {data['total_documents']}")
        else:
            st.warning("Paste some text first.")

st.divider()

st.subheader("🔍 Step 2 — Check a claim")
claim = st.text_area("Enter a claim or LLM output to fact-check", height=100,
                      placeholder="e.g. The Eiffel Tower is 500 meters tall.")

col1, col2 = st.columns([1, 3])
with col1:
    top_k = st.number_input("Top-K evidence", min_value=1, max_value=10, value=4)

if st.button("Check for hallucination", type="primary"):
    if not claim.strip():
        st.warning("Enter a claim first.")
    else:
        with st.spinner("Retrieving evidence and judging..."):
            resp = requests.post(f"{API_URL}/check", json={"claim": claim, "top_k": int(top_k)})
            result = resp.json()

        verdict = result.get("verdict", "UNCERTAIN")
        confidence = result.get("confidence", 0.0)

        color = {"GROUNDED": "green", "HALLUCINATED": "red", "UNCERTAIN": "orange"}.get(verdict, "gray")
        emoji = {"GROUNDED": "✅", "HALLUCINATED": "🚨", "UNCERTAIN": "❓"}.get(verdict, "❔")

        st.markdown(f"### {emoji} :{color}[{verdict}]  (confidence: {confidence:.2f})")
        st.write(result.get("reasoning", ""))

        supporting = result.get("supporting_evidence", [])
        if supporting:
            st.markdown("**Supporting evidence cited by judge:**")
            for s in supporting:
                st.markdown(f"- {s}")

        evidence_used = result.get("evidence_used", [])
        if evidence_used:
            with st.expander(f"📄 Raw retrieved chunks ({len(evidence_used)})"):
                for e in evidence_used:
                    st.markdown(f"**{e['source']}** — relevance: `{e['relevance_score']}`")
                    st.text(e["text"])
                    st.markdown("---")

In [ ]:
# Get your public IP (needed as the localtunnel password) and launch the tunnel.
!wget -q -O - https://loca.lt/mytunnelpassword

In [ ]:
import subprocess, threading

def run_streamlit():
    subprocess.run(["streamlit", "run", "streamlit_app.py", "--server.port", "8501", "--server.headless", "true"])

st_thread = threading.Thread(target=run_streamlit, daemon=True)
st_thread.start()
time.sleep(6)
print("Streamlit running locally on port 8501. Starting tunnel...")

# localtunnel needs npx (comes with Node, preinstalled on Colab)
!npx --yes localtunnel --port 8501

**Using the tunnel:** click the `https://*.loca.lt` URL printed above, then paste the IP address from the cell before it into the "Tunnel Password" field it asks for. Leave this cell running — closing it kills the tunnel.

### Next steps for your GitHub repo
- Split `app.py` and `streamlit_app.py` out of this notebook into your repo structure (`backend/app.py`, `frontend/streamlit_app.py`)
- Swap `chromadb.Client()` for `chromadb.PersistentClient(path="./chroma_db")` so the index survives restarts
- Add a `requirements.txt` from the installs in Part 1
- Add a `.env.example` for `ANTHROPIC_API_KEY` and load it with `python-dotenv` instead of `getpass` outside Colab